# RL Day 1：MDP / Bellman 方程 / 价值迭代

> 2026-04-22 · 阶段 2 强化学习第 1 天
>
> **目标**：从零建立 RL 的数学骨架。不写代码，不追求细节，**追求在脑海里把地图画清楚**。
>
> **教学风格**：先讲问题 → 再讲抽象 → 再讲方法。每一步都问"为什么非要这样"。


## 0. 今天要钉下的 map 节点

学完 Day 1 你脑子里应该有这些东西，且能**口头复述它们之间的关系**：

1. **RL 在解决什么问题** —— 为什么监督学习不够？
2. **MDP 五元组** $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
3. **Markov 性质** —— 为什么必须假设它
4. **Return** $G_t$ 和为什么要折扣 $\gamma$
5. **价值函数** $V^\pi$ / $Q^\pi$ / $V^*$ / $Q^*$ —— 四个量的区别
6. **Bellman Expectation Equation** —— 给定策略怎么评估它
7. **Bellman Optimality Equation** —— 最优策略满足什么
8. **价值迭代** —— 怎么迭代求解 $V^*$
9. **Contraction mapping** —— 为什么迭代保证收敛

下面一个一个来。


## 1. RL 在解决什么问题？

你刚刚训完一个 GPT。那是**监督学习**：有 $(x, y)$ 配对的数据集，你让模型最小化 $\mathcal{L}(f(x), y)$。

RL 不一样。RL 的场景是：

> **一个 agent 不断与环境交互，每次动作会改变环境状态，并得到一个反馈信号（reward）。目标：找到一个策略，让累计 reward 最大。**

举例：

| 场景 | agent | 环境 | 动作 | reward |
|---|---|---|---|---|
| 下棋 | 你 | 棋盘 | 走一步 | 赢 +1 / 输 -1（终局才给） |
| 机器人走路 | 策略网络 | 物理世界 | 关节力矩 | 前进 +1 / 摔倒 -100 |
| LLM RLHF | 语言模型 | 人类评价器 | 生成下一个 token | 人类打分 |

**RL 和监督学习的根本区别：**

| | 监督学习 | RL |
|---|---|---|
| 训练信号 | 明确的 label | reward（可能稀疏、延迟） |
| 样本独立性 | iid 假设 | 当前状态依赖历史决策（non-iid） |
| 目标 | 拟合一个函数 | 最大化长期累计收益 |
| 优化对象 | 网络权重 | 策略（同样是网络权重，但通过环境交互） |

**关键难点**：如果你下了 50 步棋最后赢了，**哪一步是关键？** reward 来得又晚又稀疏，要把它分摊到每个动作上 —— 这叫 **credit assignment problem**，是 RL 最核心的难题。


## 2. 把问题抽象成数学：MDP

为了讲数学，必须先把"agent 和环境交互"这件事形式化。标准框架叫 **Markov Decision Process (MDP)**，由五个东西定义：

$$\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$$

| 符号 | 含义 | 机器人的例子 |
|---|---|---|
| $\mathcal{S}$ | 状态空间（所有可能的状态） | 关节角度 + 速度 + 位置 |
| $\mathcal{A}$ | 动作空间（所有可能的动作） | 每个关节的力矩 |
| $P(s' \mid s, a)$ | 转移概率：在状态 $s$ 做动作 $a$，下一状态是 $s'$ 的概率 | 由物理定律决定（仿真器给） |
| $R(s, a)$ | 奖励函数：在 $s$ 做 $a$ 得到的即时 reward | 前进+1，摔倒-100 |
| $\gamma \in [0, 1)$ | 折扣因子 | 未来 reward 的贬值率 |

**agent 和 MDP 的交互循环**：

$$s_0 \xrightarrow{a_0} \underbrace{r_1, s_1}_{\text{环境给}} \xrightarrow{a_1} r_2, s_2 \xrightarrow{a_2} \dots$$

每个时刻：
1. agent 看到状态 $s_t$
2. agent 选动作 $a_t$（根据**策略** $\pi$）
3. 环境按 $P$ 转移到 $s_{t+1}$，按 $R$ 给出 $r_{t+1}$
4. 循环

**策略** $\pi(a \mid s)$：状态到动作的（概率）分布。这就是我们要学的东西 —— **RL 的核心就是找一个好的 $\pi$**。


## 3. Markov 性质：为什么必须假设它

"Markov" 这个词不是随便叫的。它指的是这样一个**强假设**：

$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \dots, s_0, a_0) = P(s_{t+1} \mid s_t, a_t)$$

翻译：**下一状态只依赖当前状态和当前动作，不依赖任何历史。**

**为什么需要这个假设？**

如果不假设它，转移概率 $P$ 的自变量就是"整条历史轨迹"，状态空间指数爆炸，无法建模。Markov 性质让我们可以只存一个"当前状态"就概括所有信息。

**代价是什么？** 很多真实任务**不满足** Markov 性质：

- 看一张 Atari 游戏截图：球的位置知道了，但**速度**呢？单帧图像缺失历史信息。
- 对话系统：当前一句话的意图往往依赖前几轮对话。
- 机器人传感器只给部分观测（比如视觉盲区）。

**解决方案**（后面会学）：
- 把历史塞进状态里（比如堆叠 4 帧图像 → 隐含速度）
- 用 RNN / Transformer 维护一个 hidden state 作为"压缩过的历史"
- 数学上这叫 **POMDP**（Partially Observable MDP）

**现在先在 MDP 框架里建直觉**。等这里通了，POMDP 只是在前面加了一层 encoder。


## 4. Return：我们到底在最大化什么？

agent 得到一串 reward：$r_1, r_2, r_3, \dots$

怎么把它们合并成一个标量目标？最自然的想法是求和：

$$G_t = r_{t+1} + r_{t+2} + r_{t+3} + \dots$$

但这有两个问题：

1. 如果任务没有终止，这个级数可能**发散**（$\infty$ 无法比较）。
2. 直觉上："明天的 100 块" 和 "10 年后的 100 块" 不该一样值钱。

所以引入**折扣因子** $\gamma \in [0, 1)$：

$$\boxed{\; G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \;}$$

这叫 **discounted return**（折扣累计奖励）。

**为什么要 $\gamma$？三个层次的理由：**

1. **数学层**：$|r| \leq r_{\max}$ 时，$|G_t| \leq r_{\max} / (1 - \gamma) < \infty$ —— 级数保证收敛。
2. **哲学层**：未来不确定性高，远期 reward 该打折。
3. **工程层**：$\gamma$ 是一个 hyperparameter：
   - $\gamma \to 0$：agent 只顾眼前（近视）
   - $\gamma \to 1$：agent 极度重视长远（远见）
   - 常用：$\gamma = 0.99$

**小记**：$\gamma$ 还有第四个身份 —— **数学上的正则化器**，让后面的 Bellman 算子成为压缩映射（下面讲）。


## 5. 价值函数：把 return 挂到状态上

$G_t$ 是**一条具体轨迹**的累计 return。但轨迹是随机的（策略随机 + 环境随机），所以我们关心**期望**。

**状态价值函数** $V^\pi(s)$：从状态 $s$ 出发，按策略 $\pi$ 走下去，能拿到多少期望 return？

$$V^\pi(s) = \mathbb{E}_\pi \left[ G_t \mid S_t = s \right]$$

**动作价值函数** $Q^\pi(s, a)$：在状态 $s$ 先采取动作 $a$（不管 $\pi$ 会怎么选），**之后**按 $\pi$ 走，能拿多少期望 return？

$$Q^\pi(s, a) = \mathbb{E}_\pi \left[ G_t \mid S_t = s, A_t = a \right]$$

**两者关系**：

$$V^\pi(s) = \sum_a \pi(a \mid s) \, Q^\pi(s, a)$$

$V$ 是 $Q$ 按策略求期望得到的。

**你必须熟记四个量**：

| 量 | 含义 | 什么时候用 |
|---|---|---|
| $V^\pi(s)$ | 策略 $\pi$ 下状态 $s$ 的价值 | 评估一个给定策略 |
| $Q^\pi(s, a)$ | 策略 $\pi$ 下状态 $s$ 选动作 $a$ 的价值 | 评估策略下每个动作 |
| $V^*(s) = \max_\pi V^\pi(s)$ | **最优**状态价值 | 找最优策略用 |
| $Q^*(s, a) = \max_\pi Q^\pi(s, a)$ | **最优**动作价值 | 直接读出最优动作 |

记住这个直觉：$V$ 是"这个局面值多少钱"，$Q$ 是"在这个局面做这件事值多少钱"。


## 6. Bellman 方程：RL 的 F=ma

这是 RL 最重要的一个式子。**整个 RL 都建立在它上面**。

**核心观察**：$V^\pi(s)$ 有一个**递归结构**。

从定义出发：

$$V^\pi(s) = \mathbb{E}_\pi [r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \dots \mid S_t = s]$$

把 $\gamma$ 之后的一堆 group 起来，用 $G_{t+1}$ 表示：

$$V^\pi(s) = \mathbb{E}_\pi [r_{t+1} + \gamma G_{t+1} \mid S_t = s]$$

而 $\mathbb{E}[G_{t+1} \mid S_{t+1} = s']$ 按定义就是 $V^\pi(s')$。展开期望：

$$\boxed{\; V^\pi(s) = \sum_a \pi(a \mid s) \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V^\pi(s') \right] \;}$$

这就是 **Bellman Expectation Equation**。

**翻译成人话**：

> 当前状态的价值 = 期望即时奖励 + 折扣后的"下一状态期望价值"

**这个等式的魔法在哪里？** 它把"整条轨迹的累计 return"这个**全局优化问题**，变成了"每对相邻状态要满足一个等式"这个**局部一致性约束**。

这个 trick 的威力怎么强调都不过分：
- 它让"评估一个策略"变成了**解一个线性方程组**。
- 它让"找最优策略"变成了**不动点迭代**（下面讲）。
- 所有的 RL 算法（DQN、PPO、SAC、Dreamer）都是在这个等式基础上长出来的。


## 7. Bellman Optimality Equation：最优策略满足什么

上面的 Bellman Expectation 是**给定策略** $\pi$ 成立的。现在我们想找**最优**策略。

**定义**：如果存在一个策略 $\pi^*$，使得**对所有状态** $s$ 都满足 $V^{\pi^*}(s) \geq V^\pi(s)$（对任意 $\pi$），则 $\pi^*$ 是最优策略。可以证明：这样的 $\pi^*$ **总是存在**（在 finite MDP 里）。

对应的最优价值函数：

$$V^*(s) = \max_\pi V^\pi(s)$$

**把 Bellman Expectation 里的 $\sum_a \pi(a|s)$ 换成 $\max_a$**，就得到：

$$\boxed{\; V^*(s) = \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V^*(s') \right] \;}$$

这叫 **Bellman Optimality Equation**。

**关键区别**：
- Expectation 是**线性**方程（$\sum \pi(a|s)$ 是线性操作）。
- Optimality 是**非线性**方程（$\max$ 是非线性操作）。
- 后者更难解，但它定义了 RL 的"圣杯"—— 只要解出 $V^*$，问题就基本完结。

**怎么从 $V^*$ 拿最优策略？**

$$\pi^*(s) = \arg\max_a \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V^*(s') \right]$$

**⚠️ 注意**：这个式子里有 $R(s,a)$ 和 $P(s'|s,a)$。意思是：**必须知道环境模型**才能从 $V^*$ 反推出 $\pi^*$。

如果你学的是 $Q^*(s, a)$ 而不是 $V^*(s)$，提取策略就简单得多：

$$\pi^*(s) = \arg\max_a Q^*(s, a)$$

只需要 argmax，**不需要 $R$，也不需要 $P$**。这就是后面 Q-Learning 学 $Q^*$ 不学 $V^*$ 的根本原因 —— **model-free**。先记住这一点，下一课就用。


## 7.5 澄清：期望展开式 & Expectation vs Optimality

（2026-04-22 补充。针对两个公认绕的点再打磨一遍。）

---

### (A) 期望展开式到底在算啥

先把目标式子摆这儿：

$$V^\pi(s) = \sum_a \pi(a \mid s) \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V^\pi(s') \right]$$

两层求和 + 嵌套 $V^\pi$，看着头疼。**但它只是在做一件事**：把期望 $\mathbb{E}_\pi[r_{t+1} + \gamma V^\pi(s_{t+1}) \mid s_t = s]$ 写成显式的加权求和。

**关键观察**：从状态 $s$ 出发，"下一步会发生什么"有**两个随机源**：

1. **策略随机**：agent 按 $\pi(a \mid s)$ 随机挑一个动作 $a$
2. **环境随机**：选定 $a$ 后，环境按 $P(s' \mid s, a)$ 随机跳到某个 $s'$

要算期望，就得对这两个随机源**同时**求加权平均。画个树会清楚很多：

```
              s  （当前状态）
             /|\
            / | \           ← 第一层分支：策略 π(a|s)  选动作
           a1 a2 a3
          /|\ ...            ← 第二层分支：环境 P(s'|s,a)  转状态
         / | \
        s' s' s'             ← 每个叶子贡献 R(s,a) + γV(s')
```

每条"从根到叶"的路径发生概率 = $\pi(a \mid s) \cdot P(s' \mid s, a)$。
每条路径贡献的"立即回报 + 未来价值" = $R(s, a) + \gamma V^\pi(s')$。

**把所有路径按概率加权相加，就是 $V^\pi(s)$**：

$$V^\pi(s) = \underbrace{\sum_a \pi(a \mid s)}_{\text{对策略求期望}} \; \underbrace{\Big[ R(s, a) + \gamma \underbrace{\sum_{s'} P(s' \mid s, a)}_{\text{对环境求期望}} V^\pi(s') \Big]}_{\text{给定 }a\text{ 之后的条件期望}}$$

**为什么 $R(s,a)$ 被"夹在中间"、不受 $\sum_{s'}$ 作用？**
因为按标准 MDP 约定，$R$ 只依赖 $(s, a)$，不依赖 $s'$。所以选定 $a$ 之后，$R(s,a)$ 是一个**定值**，不用再对 $s'$ 求期望。只有 $V^\pi(s')$ 才依赖 $s'$，所以 $\sum_{s'}$ 只套在 $V^\pi(s')$ 外面。

> **一句话记忆**：外层 $\sum_a \pi$ 是 "策略抽动作"，内层 $\sum_{s'} P$ 是 "环境抽下一状态"。中间那个 $R(s,a)$ 是选完动作后的"定金"。

---

### (B) Expectation vs Optimality：一个在评估，一个在优化

这俩等式长得像双胞胎，唯一差别就一个符号：

| | Bellman **Expectation** | Bellman **Optimality** |
|---|---|---|
| 形式 | $V^\pi(s) = \sum_a \pi(a\mid s) [\dots]$ | $V^*(s) = \max_a [\dots]$ |
| 在回答什么问题 | **给一个策略 $\pi$，它值多少钱？** | **最好的策略值多少钱？** |
| RL 术语 | **policy evaluation**（策略评估） | **control / optimization**（最优控制） |
| $\pi$ 在哪？ | 显式出现在等式里 | **消失了** |
| 数学性质 | **线性**方程（$\sum$ 是线性操作） | **非线性**方程（$\max$ 非线性） |
| 解法 | 解一个线性方程组（closed form 存在） | 不动点迭代（value iteration） |

**核心直觉**：`Σ_a π(a|s)` 和 `max_a` 都是"对动作做聚合"的操作，但聚合方式不同 ——

- `Σ_a π(a|s) [...]` = **按策略给的概率加权平均**（"这个策略到底会选啥我说了不算，我只能算期望"）
- `max_a [...]` = **直接挑最大的**（"我说了算，我永远挑最好的那个动作"）

**为什么 Optimality 里 $\pi$ 消失了？**

因为最优策略有一个漂亮性质 —— **它一定是确定性的**（在标准 MDP 下）。"最优"就意味着每个状态下永远挑一个最好的动作，不需要概率分布。所以 $\pi^*$ 天然就是 argmax：

$$\pi^*(s) = \arg\max_a \Big[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V^*(s') \Big]$$

当把这个"argmax 策略"代回 Bellman Expectation 里，$\sum_a \pi^*(a\mid s) [\dots]$ 就坍缩成 $\max_a [\dots]$ —— 因为 $\pi^*$ 只在最大的那个 $a$ 上取 1，其他都取 0。

**这也是两个方程难度差的来源**：
- Expectation：$\pi$ 是**已知**的，等式是线性的 → 在 $|\mathcal{S}|$ 维空间里是一个线性方程组，可以 $(I - \gamma P^\pi)^{-1}$ 一次解完。
- Optimality：相当于要在"所有可能的 $\pi$"里挑最好的，外面套了一层优化 → 非线性 → 只能迭代求解。

---

### (C) 两个问题对应两类 RL 算法

这个对应关系**非常重要**，后面整个 RL 课程都在这张表里打转：

| 问题 | 方程 | 典型算法 |
|---|---|---|
| 我有一个策略，帮我评估它 | Bellman Expectation | **Policy Evaluation** / TD(0) / Monte Carlo |
| 我要找最优策略 | Bellman Optimality | **Value Iteration** / Q-Learning / DQN |
| 我既想评估又想改进 | 两者交替 | **Policy Iteration** / Actor-Critic / PPO |

记住这张表，后面每学一个新算法都可以问自己："这个算法是在解 Expectation 还是 Optimality？怎么解的？"

---

### (D) 自查

这两段搞懂了应该能答：

- [ ] 为什么期望展开式里有两层 $\sum$？分别对应什么随机源？
- [ ] 为什么 $R(s, a)$ 没被 $\sum_{s'}$ 套起来？
- [ ] Bellman Expectation 和 Optimality 在"解决什么问题"上有什么区别？
- [ ] 为什么 Optimality 等式里看不到 $\pi$ 了？
- [ ] 为什么 Expectation 有 closed-form 解但 Optimality 没有？


## 8. 价值迭代：怎么求解 Bellman Optimality Equation

Bellman Optimality 是个递归的非线性方程，直接解不动。但它有个漂亮的性质：**可以用不动点迭代求解**。

**算法**（value iteration）：

```
初始化  V(s) ← 0  for all s ∈ S
重复直到收敛：
    for each s ∈ S:
        V(s) ← max_a [ R(s,a) + γ Σ_{s'} P(s'|s,a) · V(s') ]
```

也就是说，**把 Bellman Optimality 等式当成赋值语句反复用**。直观感觉很不靠谱 —— 为什么这就能收敛到 $V^*$？

因为下面这个定理。


## 9. 为什么价值迭代一定收敛：Contraction Mapping

定义 **Bellman Optimality Operator** $\mathcal{T}^*$：

$$(\mathcal{T}^* V)(s) \;\triangleq\; \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V(s') \right]$$

$\mathcal{T}^*$ 是一个**把价值函数映射到价值函数的算子**。价值迭代每一步就是在算 $V_{k+1} = \mathcal{T}^* V_k$。

**关键定理**：$\mathcal{T}^*$ 在 sup-norm（$\infty$-norm）下是一个 **$\gamma$-contraction mapping**：

$$\| \mathcal{T}^* V_1 - \mathcal{T}^* V_2 \|_\infty \leq \gamma \, \| V_1 - V_2 \|_\infty$$

翻译：任何两个价值函数过一遍 $\mathcal{T}^*$，距离会缩小到原来的 $\gamma$ 倍。

**Banach 不动点定理**：完备度量空间上的 contraction mapping **有且只有一个不动点**，且从**任意**初始点开始迭代都会**指数收敛**到它。

把两个事实拼起来：

1. $V^*$ 是 $\mathcal{T}^*$ 的不动点（Bellman Optimality 就是说 $V^* = \mathcal{T}^* V^*$）。
2. $\mathcal{T}^*$ 是 contraction → 不动点唯一 → 从任意 $V_0$ 开始迭代必收敛到 $V^*$。

**收敛速度**：每轮误差乘 $\gamma$。$\gamma = 0.99$ 时，约 460 轮能把误差降到 1%。

**这就是为什么 $\gamma < 1$ 在数学上是必要的**：如果 $\gamma = 1$，contraction 性质破坏，收敛不再保证。这是折扣因子**第四个身份**：数学正则化器。


## 10. 这个框架会被"推翻"的部分（现代 RL 为什么要超越它）

MDP + Bellman + 价值迭代是 RL 的数学骨架。但这套"经典 RL"有三个软肋，驱动了后面所有发展：

| 软肋 | 问题 | 解决方向 |
|---|---|---|
| **需要知道 $P(s' \mid s, a)$** | 真实世界物理规律没有 closed-form | **Q-Learning**（model-free，直接从经验学 $Q$） |
| **状态空间必须离散且小** | 机器人 1000 维连续状态没法枚举 | **DQN**（神经网络近似 $V$/$Q$） |
| **Markov 假设太强** | 部分可观测任务不适用 | **POMDP + RNN/Transformer** 策略 |

**但是！这个框架的数学骨架永远不过时**：DQN、PPO、SAC、Dreamer、RLHF —— 都是从 Bellman 方程长出来的。所以必须先把它吃透。

这也是为什么我们 Day 1 花一整天讲 MDP 和 Bellman —— **它是地基**。


## 11. Day 1 的 map 节点 —— 自测

看下面这些问题。如果你能**不查资料、用自己的话**答出来，Day 1 就过了：

- [ ] MDP 五元组是哪五个东西？各自是什么意思？
- [ ] 什么叫 Markov 性质？为什么 RL 要假设它？
- [ ] $G_t$ 的定义是什么？为什么要折扣 $\gamma$？
- [ ] $V^\pi$、$Q^\pi$、$V^*$、$Q^*$ 四个量的区别？
- [ ] Bellman Expectation 和 Bellman Optimality 的区别？
- [ ] 为什么说 Bellman 方程把"全局优化"变成了"局部一致性"？
- [ ] 价值迭代是什么？为什么它保证收敛？
- [ ] 如果我已经有了 $V^*$，为什么还需要模型才能拿到 $\pi^*$？$Q^*$ 为什么不需要？

答不出来的话，回到对应章节再看一遍。


## 12. 三个思考题（今天的压轴）

这三个题不需要写代码，但每个都在考一个**核心概念**。想清楚之后跟我讨论。

**Q1.** 为什么 $\gamma$ 必须 $< 1$？如果轨迹有限长（比如回合制游戏到终局就停），$\gamma = 1$ 行不行？

**Q2.** 看 Bellman Optimality Equation。如果我**已经知道 $V^*$**，怎么在**不做任何学习**的情况下拿到最优策略？这给了你什么启示（为什么 Q-Learning 后面要学 $Q^*$ 而不是 $V^*$）？

**Q3.** 考虑一个 gridworld，所有非目标格子 reward $= -1$（鼓励尽快到达目标），目标格子 reward $= 0$，$\gamma = 0.9$。如果我把所有 reward 整体 $+1$（变成非目标 $0$、目标 $+1$），最优策略会变吗？会变的话怎么变？这叫 **reward shaping** 的一个特例。

想清楚之后告诉我你的答案，我们一起看哪里卡住。
